In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL='gpt-6-luna'

#### Images and vision
- vision : 모델이 이미지를 입력으로 받아 이해하는 능력
- response api 를 이용
- image api 이용

In [ ]:
# 이미지 생성
from openai import OpenAI
import base64

client = OpenAI()

response = client.responses.create(
    model=MODEL,
    input="Generate an image of gray tabby cat hugging an otter with an orange scarf",
    tools=[{"type":"image_generation"}]
)

# 생성된 이미지 저장
image_data = [output.result for output in response.output if output.type == "image_generation_call"]

if image_data:
    image_base64 = image_data[0]
    with open("./data/cat_and_otter.png","wb") as f:
        f.write(base64.b64decode(image_base64))

In [4]:
# 웹 이미지 분석

content = [
    {"type":"input_text", "text":"what's in this image?"},
    {"type":"input_image", "image_url":"https://img.khan.co.kr/news/r/700xX/2026/01/07/news-p.v1.20260107.29cd9ce51ca44c718998d177e9c0fa07_P1.webp"},
]

response = client.responses.create(
    model=MODEL,
    input=[
        {"role":"user", "content":content}
    ],
    tools=[{"type":"image_generation"}]
)

print(response.output_text)

Three giant pandas are relaxing around a bamboo-and-flower display: one sits facing the camera, another peeks from behind it, and a third lies on its back in the foreground holding bamboo. The display has signs reading “RUIBAO” and “HUIBAO.”


In [7]:
# 로컬 이미지 파일 분석

# b : text 파일 아닌경우
def encode_image(image_path):
    with open(image_path,"rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [8]:
image_path = "./data/bag.png"

# 이미지 로드
base64_image = encode_image(image_path)

content = [
    {"type":"input_text", "text":"what's in this image?"},
    {"type":"input_image", "image_url":f"data:image/jpeg;base64,{base64_image}"},
]

response = client.responses.create(
    model=MODEL,
    input=[
        {"role":"user", "content":content}
    ]
)

print(response.output_text)


A dark green duffel bag with brown handles and an adjustable shoulder strap. It has a zippered bottom compartment and a small front pocket.


In [ ]:
# file id 이용
# purpose="vision"  => 파인튜닝 용도
# purpose="user_data" => 일반적인 용도

def create_file(file_path):
    with open(file_path,"rb") as f:
        result = client.files.create(file=f, purpose="user_data")
        return result.id

In [12]:
image_path = "./data/bag.png"

# 이미지 로드
file_id = create_file(image_path)

content = [
    {"type":"input_text", "text":"what's in this image?"},
    {"type":"input_image", "file_id":file_id},
]

response = client.responses.create(
    model=MODEL,
    input=[
        {"role":"user", "content":content}
    ]
)

print(response.output_text)

A dark green duffel bag with tan handles and an adjustable shoulder strap, black trim, and a zippered bottom compartment.


#### 이미지 전용 모델

In [14]:
response = client.images.generate(
    model="gpt-image-2",
    prompt="A cute red panda reading a book, watercolor style",
    size="auto", # 1024x1024, 1024x1536, 1536x1024
    quality="high",  # low, medium, high, auto
    output_format="png"
)

image_base64 = response.data[0].b64_json
imgae_bytes = base64.b64decode(image_base64)

with open("./data/panda.png","wb") as f:
    f.write(imgae_bytes)

#### Audio and Speech
- 음성 비서
- 실시간 음성 처리
- TTS(Text-to-Speech)
- STT(Speech -to-Text)

In [25]:
# TTS
speech = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="verse",
    input="AI 인공지능 TTS 입니다람쥐",
    speed=1.5,
    response_format="mp3"
)

with open("./data/tts1.mp3","wb") as f:
    f.write(speech.read())

In [21]:
# STT 음성 => 텍스트

with open("./data/speech.mp3","rb") as f:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=f       
    )

print(transcription.text)

안녕하세요. 이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다. GPT API에 대해서 생소하신 분들도 있을 텐데, 우리가 잘 알고 있는 챗GPT, 챗GPT의 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러 가지들이 있는데, 좀 이 강의의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챗봇 프로그램만 만드는 게 사실 쉽지는 않은데, 이거를 어떻게 해서 구현을 하는지, 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로, 예제는 여러 가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고.


In [22]:
with open("./data/rag.mp3","rb") as f:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=f,
        language="en",
        prompt="AI 개발 관련 강의이다." \
        "주요 용어 :" \
        "LLM, RAG"       
    )

print(transcription.text)

Large language models, they are everywhere. They get some things amazingly right and other things very interestingly wrong. My name is Marina Danilevsky. I am a senior research scientist here at IBM Research, and I want to tell you about a framework to help large language models be more accurate and more up to date: retrieval augmented generation, or RAG. Let's just talk about the generation part for a minute. So forget the retrieval augmented. So the generation, this refers to large language models, or LLMs, that generate text in response to a user query, referred to as a prompt. These models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: In our solar system, what planet has the most moons? And my response was, Oh, that's really great that you're asking me this question. I loved space when I was your age. Of course, that was like 30 years ago. But I know this. I read an article, and the article s

In [24]:
# 음성을 문장별로 분리 / 시간 정보

with open("./data/rag.mp3","rb") as f:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="verbose_json",
        timestamp_granularities=['segment']
    )

for segment in transcription.segments:
    print(f"{segment.start:.2f} ~ {segment.end:.2f}", segment.text)

0.00 ~ 2.92  Large language models, they are everywhere.
2.92 ~ 5.32  They get some things amazingly right
5.32 ~ 8.20  and other things very interestingly wrong.
8.20 ~ 9.66  My name is Marina Danilevsky.
9.66 ~ 12.44  I am a senior research scientist here at IBM Research.
12.44 ~ 14.80  And I want to tell you about a framework
14.80 ~ 17.48  to help large language models be more accurate
17.48 ~ 18.92  and more up-to-date.
18.92 ~ 22.10  Retrieval Augmented Generation, or RAG.
22.92 ~ 24.84  Let's just talk about the generation part for a minute.
24.84 ~ 26.92  So forget the retrieval augmented.
26.92 ~ 30.28  So the generation, this refers to large language models
30.28 ~ 34.74  or LLMs that generate text in response to a user query
34.74 ~ 36.24  referred to as a prompt.
36.24 ~ 38.36  These models can have some undesirable behavior.
38.36 ~ 41.28  I want to tell you an anecdote to illustrate this.
41.28 ~ 44.72  So my kids, they recently asked me this question.
44.72 ~ 49.24  In o

In [26]:
# 1. 음성 파일을 텍스트로 변환 : chunk.mp3
with open("./data/chunk.mp3","rb") as f:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=f,        
    )

text = transcription.text
print(text)

# 2. 텍스트로 변환한 내용을 기반으로 답변 생성
response = client.responses.create(
    model=MODEL,
    instructions="당신은 친절한 AI 비서입니다. 간결하고 자연스럽게 답변하세요",
    input=text
)
answer = response.output_text
print("AI",answer)

# 3. 답변을 음성으로 변환 answer.mp3
speech = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=answer,   
)

with open("./data/answer.mp3","wb") as f:
    f.write(speech.read())

첫 번째 이해해야 되는 개념이 바로 청크입니다. 그래서 문서를 나누는 게 바로 청크이죠. 긴 문서를 검색 증강 생성, RAG에서 다루기 좋게 적당한 길이로 의미 단위로 자르는 것, 바로 이 청크가 되겠습니다. 일반적으로는 한 200에서 한 800 토큰 정도로 우리가 이 청크를 만들게 되고요. 그리고 단락이나 섹션이나 표 이런 문서 구조를 유지한 상태로 나누는 것이 이상적이다라고 얘기를 하죠. 그리고 문서 내에 단락이나 섹션들이 있을 겁니다. 그래서 의미가 사실은 유지가 되어야 되기 때문에, 그리고 단락별로, 문장별로, 크게는 섹션별로 청크를 하게 되면은 의미가 좀 더 정확하게 유지가 되겠죠.
AI 맞습니다. 청크는 긴 문서를 RAG에서 검색하고 활용하기 쉽도록 의미 단위로 나눈 조각입니다. 보통 200~800토큰 정도를 기준으로 삼지만, 문서의 성격과 검색 목적에 따라 달라질 수 있습니다. 단락·섹션·표 같은 구조를 최대한 유지하고, 문맥이 끊기지 않도록 청크 사이에 일부 내용을 겹치게 하기도 합니다.
